In [1]:
import sqlite3
import pandas as pd

In [3]:
df = pd.read_csv('D:\\KirtiDA_Prep\\Projects\\telecom-network-analytics\\data\\cleaned\\network_performance_cleaned.csv')
print("Data loaded successfully")
print(df.shape)
df.head()

Data loaded successfully
(3605, 11)


,timestamp,tower_id,users_connected,download_speed,upload_speed,latency,weather,congestion,date,hour,day_name
0,2025-01-01 00:00:00,1,152,80.671584,9.988305,158.141290,Clear,0,2025-01-01,0,Wednesday
1,2025-01-01 01:00:00,1,171,19.819479,3.846097,174.573468,Clear,0,2025-01-01,1,Wednesday
2,2025-01-01 02:00:00,1,713,66.834405,3.764167,147.179767,Snow,0,2025-01-01,2,Wednesday
3,2025-01-01 03:00:00,1,435,22.273372,9.986821,67.806026,Clear,0,2025-01-01,3,Wednesday
4,2025-01-01 04:00:00,1,797,7.190930,26.713958,85.973585,Clear,0,2025-01-01,4,Wednesday


In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df.dtypes)

timestamp          datetime64[ns]
tower_id                    int64
users_connected             int64
download_speed            float64
upload_speed              float64
latency                   float64
weather                    object
congestion                  int64
date                       object
hour                        int64
day_name                   object
dtype: object


In [5]:
conn = sqlite3.connect("telecom_network.db")
print("SQLite connection created")

SQLite connection created


In [6]:
df.to_sql(
    "network_performance",
    conn,
    if_exists="replace",
    index=False
)

print("Table created successfully")

Table created successfully


In [7]:
query = """
SELECT *
FROM network_performance
LIMIT 10;
"""

sql_preview = pd.read_sql_query(query, conn)

sql_preview

,timestamp,tower_id,users_connected,download_speed,upload_speed,latency,weather,congestion,date,hour,day_name
0,2025-01-01 00:00:00,1,152,80.671584,9.988305,158.141290,Clear,0,2025-01-01,0,Wednesday
1,2025-01-01 01:00:00,1,171,19.819479,3.846097,174.573468,Clear,0,2025-01-01,1,Wednesday
2,2025-01-01 02:00:00,1,713,66.834405,3.764167,147.179767,Snow,0,2025-01-01,2,Wednesday
3,2025-01-01 03:00:00,1,435,22.273372,9.986821,67.806026,Clear,0,2025-01-01,3,Wednesday
4,2025-01-01 04:00:00,1,797,7.190930,26.713958,85.973585,Clear,0,2025-01-01,4,Wednesday
5,2025-01-01 05:00:00,1,749,39.804375,23.347429,159.183433,Clear,0,2025-01-01,5,Wednesday
6,2025-01-01 06:00:00,1,616,98.406934,23.871382,173.388677,Clear,0,2025-01-01,6,Wednesday
7,2025-01-01 07:00:00,1,890,11.179901,47.495391,193.470086,Rain,1,2025-01-01,7,Wednesday
8,2025-01-01 08:00:00,1,826,6.516794,12.313797,55.794839,Clear,0,2025-01-01,8,Wednesday
9,2025-01-01 09:00:00,1,477,52.041806,2.685038,182.770876,Clear,0,2025-01-01,9,Wednesday


In [8]:
query = """
SELECT COUNT(*) AS total_records
FROM network_performance;
"""

pd.read_sql_query(query, conn)

,total_records
0,3605


In [9]:
# Average performance by tower
query = """
SELECT
    tower_id,
    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(download_speed), 2) AS avg_download_speed,
    ROUND(AVG(upload_speed), 2) AS avg_upload_speed,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY tower_id
ORDER BY tower_id;
"""

tower_sql = pd.read_sql_query(query, conn)

tower_sql

,tower_id,avg_users,avg_download_speed,avg_upload_speed,avg_latency,congestion_rate
0,1,516.85,52.71,25.37,105.38,5.41
1,2,544.75,50.79,24.38,106.62,6.24
2,3,524.63,52.39,25.20,102.47,4.58
3,4,537.87,52.38,25.62,105.27,6.66
4,5,531.36,52.18,25.27,105.77,5.27


In [10]:
# Tower with the highest latency
query = """
SELECT
    tower_id,
    ROUND(AVG(latency), 2) AS avg_latency
FROM network_performance
GROUP BY tower_id
ORDER BY avg_latency DESC
LIMIT 1;
"""

highest_latency_sql = pd.read_sql_query(query, conn)

highest_latency_sql

,tower_id,avg_latency
0,2,106.62


In [11]:
# Tower with the highest congestion
query = """
SELECT
    tower_id,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY tower_id
ORDER BY congestion_rate DESC
LIMIT 1;
"""

highest_congestion_sql = pd.read_sql_query(query, conn)

highest_congestion_sql

,tower_id,congestion_rate
0,4,6.66


In [12]:
# Hourly network performance
query = """
SELECT
    hour,
    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(download_speed), 2) AS avg_download_speed,
    ROUND(AVG(upload_speed), 2) AS avg_upload_speed,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY hour
ORDER BY hour;
"""

hourly_sql = pd.read_sql_query(query, conn)

hourly_sql

,hour,avg_users,avg_download_speed,avg_upload_speed,avg_latency,congestion_rate
0,0,555.40,51.92,24.62,105.73,7.10
1,1,539.52,51.81,26.18,107.27,9.33
2,2,569.01,51.65,25.93,112.88,7.33
3,3,527.47,51.46,23.65,99.42,4.67
4,4,515.81,49.31,24.06,107.94,2.67
5,5,544.43,50.84,25.32,111.37,9.33
6,6,557.19,50.89,26.37,103.68,7.33
7,7,507.23,51.31,26.01,106.32,6.67
8,8,531.97,54.38,24.41,96.16,3.33
9,9,518.34,46.14,23.68,106.94,5.33


In [13]:
# Top three congestion hours
query = """
SELECT
    hour,
    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY hour
ORDER BY congestion_rate DESC
LIMIT 3;
"""

top_congestion_hours_sql = pd.read_sql_query(query, conn)

top_congestion_hours_sql

,hour,avg_users,avg_latency,congestion_rate
0,18,551.67,112.50,11.33
1,16,561.76,99.52,9.33
2,5,544.43,111.37,9.33


In [14]:
# Network performance by weather
query = """
SELECT
    weather,
    COUNT(*) AS record_count,
    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(download_speed), 2) AS avg_download_speed,
    ROUND(AVG(upload_speed), 2) AS avg_upload_speed,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY weather
ORDER BY congestion_rate DESC;
"""

weather_sql = pd.read_sql_query(query, conn)

weather_sql

,weather,record_count,avg_users,avg_download_speed,avg_upload_speed,avg_latency,congestion_rate
0,Storm,182,560.72,52.59,24.33,105.60,7.69
1,Snow,340,547.87,50.68,26.25,109.71,7.35
2,Rain,541,527.19,52.96,25.02,107.81,7.02
3,Clear,2542,527.56,52.06,25.11,103.87,4.96


In [15]:
# Daily network performance
query = """
SELECT
    date,
    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(download_speed), 2) AS avg_download_speed,
    ROUND(AVG(upload_speed), 2) AS avg_upload_speed,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY date
ORDER BY date;
"""

daily_sql = pd.read_sql_query(query, conn)

daily_sql.head(10)

,date,avg_users,avg_download_speed,avg_upload_speed,avg_latency,congestion_rate
0,2025-01-01,563.06,53.84,24.29,105.76,5.00
1,2025-01-13,509.68,53.74,22.94,111.68,5.83
2,2025-01-14,505.98,53.26,23.66,104.41,10.83
3,2025-01-15,523.99,46.87,26.47,107.61,5.83
4,2025-01-16,566.14,51.32,23.65,111.01,6.67
5,2025-01-17,549.19,52.04,25.99,102.79,5.00
6,2025-01-18,534.92,53.67,24.90,103.03,3.33
7,2025-01-19,544.95,50.90,27.00,110.55,7.50
8,2025-01-20,481.88,52.67,25.34,109.27,3.33
9,2025-01-21,594.84,51.11,23.55,105.37,6.67


In [17]:
# Towers requiring investigation
query = """
SELECT
    tower_id,
    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate
FROM network_performance
GROUP BY tower_id
HAVING AVG(latency) > 105
   AND AVG(congestion) > 0.05
ORDER BY congestion_rate DESC;
"""

investigation_sql = pd.read_sql_query(query, conn)

investigation_sql

,tower_id,avg_users,avg_latency,congestion_rate
0,4,537.87,105.27,6.66
1,2,544.75,106.62,6.24
2,1,516.85,105.38,5.41
3,5,531.36,105.77,5.27


In [18]:
# Exporting the SQL results
tower_sql.to_csv("sql_tower_kpis.csv", index=False)
hourly_sql.to_csv("sql_hourly_kpis.csv", index=False)
weather_sql.to_csv("sql_weather_kpis.csv", index=False)
daily_sql.to_csv("sql_daily_kpis.csv", index=False)
investigation_sql.to_csv("sql_investigation_kpis.csv", index=False)

print("SQL KPI files exported successfully")

SQL KPI files exported successfully


In [19]:
print("Tower KPIs:")
print(tower_sql)

print("\nHighest latency tower:")
print(highest_latency_sql)

print("\nHighest congestion tower:")
print(highest_congestion_sql)

print("\nTop congestion hours:")
print(top_congestion_hours_sql)

print("\nWeather KPIs:")
print(weather_sql)

print("\nInvestigation towers:")
print(investigation_sql)

Tower KPIs:
   tower_id  avg_users  avg_download_speed  avg_upload_speed  avg_latency  \
0         1     516.85               52.71             25.37       105.38   
1         2     544.75               50.79             24.38       106.62   
2         3     524.63               52.39             25.20       102.47   
3         4     537.87               52.38             25.62       105.27   
4         5     531.36               52.18             25.27       105.77   

   congestion_rate  
0             5.41  
1             6.24  
2             4.58  
3             6.66  
4             5.27  

Highest latency tower:
   tower_id  avg_latency
0         2       106.62

Highest congestion tower:
   tower_id  congestion_rate
0         4             6.66

Top congestion hours:
   hour  avg_users  avg_latency  congestion_rate
0    18     551.67       112.50            11.33
1    16     561.76        99.52             9.33
2     5     544.43       111.37             9.33

Weather KPIs:
  weat

In [ ]:
#Findings
#In this dataset, adverse weather categories show higher congestion rates than clear weather. 
#Snow has the highest average latency at 109.71, while Storm has the highest congestion rate at 7.69%. 
#These results show an association in the dataset, not proof that weather directly caused the performance changes.

In [20]:
query = """
SELECT
    tower_id,
    hour,
    weather,

    ROUND(AVG(users_connected), 2) AS avg_users,
    ROUND(AVG(download_speed), 2) AS avg_download_speed,
    ROUND(AVG(upload_speed), 2) AS avg_upload_speed,
    ROUND(AVG(latency), 2) AS avg_latency,
    ROUND(AVG(congestion) * 100, 2) AS congestion_rate,

    COUNT(*) AS record_count

FROM network_performance

GROUP BY
    tower_id,
    hour,
    weather

ORDER BY
    tower_id,
    hour,
    weather;
"""

dashboard_data = pd.read_sql_query(query, conn)

print("Dashboard rows:", dashboard_data.shape[0])
dashboard_data.head(10)

Dashboard rows: 448


,tower_id,hour,weather,avg_users,avg_download_speed,avg_upload_speed,avg_latency,congestion_rate,record_count
0,1,0,Clear,568.83,48.18,18.87,90.90,0.00,23
1,1,0,Rain,565.33,51.56,19.18,75.86,0.00,6
2,1,0,Snow,85.00,94.57,30.34,142.01,0.00,1
3,1,0,Storm,507.00,90.26,26.80,162.14,0.00,1
4,1,1,Clear,557.32,68.40,22.04,101.99,0.00,19
5,1,1,Rain,557.17,50.45,35.19,102.61,16.67,6
6,1,1,Snow,214.00,27.16,29.48,174.00,0.00,1
7,1,1,Storm,587.75,44.13,21.53,114.21,0.00,4
8,1,2,Clear,514.32,51.21,25.72,113.73,9.09,22
9,1,2,Rain,648.50,18.50,35.49,104.17,25.00,4


In [21]:
# exporting summary to csv
dashboard_data.to_csv(
    "telecom_dashboard_data.csv",
    index=False
)

print("Dashboard file created successfully")

Dashboard file created successfully


In [22]:
df.to_csv(
    "telecom_network_final.csv",
    index=False
)

print("Original cleaned file exported")

Original cleaned file exported


In [23]:
# KPI summary table
kpi_summary = pd.DataFrame({
    "KPI": [
        "Average Users Connected",
        "Average Download Speed",
        "Average Upload Speed",
        "Average Latency",
        "Overall Congestion Rate",
        "Highest Latency Tower",
        "Highest Congestion Tower",
        "Highest Congestion Hour"
    ],
    "Value": [
        round(df["users_connected"].mean(), 2),
        round(df["download_speed"].mean(), 2),
        round(df["upload_speed"].mean(), 2),
        round(df["latency"].mean(), 2),
        round(df["congestion"].mean() * 100, 2),
        int(highest_latency_sql.iloc[0]["tower_id"]),
        int(highest_congestion_sql.iloc[0]["tower_id"]),
        int(top_congestion_hours_sql.iloc[0]["hour"])
    ]
})

kpi_summary

,KPI,Value
0,Average Users Connected,531.09
1,Average Download Speed,52.09
2,Average Upload Speed,25.17
3,Average Latency,105.10
4,Overall Congestion Rate,5.63
5,Highest Latency Tower,2.00
6,Highest Congestion Tower,4.00
7,Highest Congestion Hour,18.00


In [24]:
kpi_summary.to_csv(
    "telecom_kpi_summary.csv",
    index=False
)

print("KPI summary exported")

KPI summary exported
